# Komparasi 3 Model Machine Learning & Deep Learning: Prediksi Cuaca Hujan di Indonesia
Notebook ini disusun secara **step-by-step** untuk membandingkan 3 pendekatan model:
1. **Model 1 (ML Biasa 1 - Linear Baseline)**: `Logistic Regression`
2. **Model 2 (ML Biasa 2 - Tree-Based Ensemble)**: `Random Forest Classifier`
3. **Model 3 (Deep Learning)**: `Deep Neural Network (MLP)` dengan **TensorFlow / Keras**

Dataset yang digunakan: **Indonesia Hourly Weather for Rain Prediction** (`karanroyka/indonesia-hourly-weather-for-rain-prediction`).
Target klasifikasi: `rain_next_6h` (Apakah akan turun hujan dalam kurun waktu 6 jam ke depan: 1 = Hujan, 0 = Tidak).


---
## Step 1: Import Library & Setup Environment
Pastikan notebook ini dijalankan menggunakan kernel virtual environment (`.venv` / `Python (ml-gabut-venv)`).


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Kagglehub
import kagglehub

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# Set random seed untuk reproduksibilitas
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"TensorFlow Version: {tf.__version__}")
print("Semua dependencies siap!")


---
## Step 2: Load Dataset dari Folder Lokal (`./dataset`)
Dataset cuaca telah tersimpan di folder lokal `./dataset/` yang berisi `train.csv`, `valid.csv`, dan `test.csv`.


In [ ]:
# Path dataset lokal
data_dir = os.path.join(".", "dataset")

# Fallback ke kagglehub jika folder dataset belum ada
if not os.path.exists(os.path.join(data_dir, "train.csv")):
    print("Folder dataset lokal tidak ditemukan, mendownload ulang via kagglehub...")
    dataset_path = kagglehub.dataset_download("karanroyka/indonesia-hourly-weather-for-rain-prediction")
    data_dir = os.path.join(dataset_path, "data")

print("Menggunakan path data:", os.path.abspath(data_dir))

# Load dataset train, valid, dan test
df_train = pd.read_csv(os.path.join(data_dir, "train.csv"))
df_valid = pd.read_csv(os.path.join(data_dir, "valid.csv"))
df_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print(f"Ukuran Train : {df_train.shape}")
print(f"Ukuran Valid : {df_valid.shape}")
print(f"Ukuran Test  : {df_test.shape}")
df_train.head()


---
## Step 3: Exploratory Data Analysis (EDA)
Mari kita periksa:
- Missing values
- Distribusi kelas target `rain_next_6h`
- Korelasi antara variabel cuaca dengan target hujan


In [ ]:
# Cek missing values
print("Missing values pada Train:")
print(df_train.isnull().sum())

# Cek distribusi target
target_counts = df_train['rain_next_6h'].value_counts()
target_prop = df_train['rain_next_6h'].value_counts(normalize=True) * 100

print("\nDistribusi Target (rain_next_6h):")
for k in target_counts.index:
    label = 'Hujan (1)' if k == 1 else 'Tidak Hujan (0)'
    print(f"  {label}: {target_counts[k]:,} baris ({target_prop[k]:.2f}%)")


In [ ]:
# Visualisasi Distribusi Target
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='rain_next_6h', hue='rain_next_6h', data=df_train, ax=ax, palette=['#4A90E2', '#E94E77'], legend=False)
ax.set_title("Distribusi Target: Hujan dalam 6 Jam ke Depan", fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Tidak Hujan (0)', 'Hujan (1)'])
ax.set_ylabel("Jumlah Observasi")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", 
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()


In [ ]:
# Korelasi Fitur Numerik Cuaca terhadap Target
numeric_cols = [
    'temperature_2m', 'relative_humidity_2m', 'apparent_temperature',
    'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',
    'precipitation', 'rain', 'rain_sum_next_6h', 'rain_next_6h'
]

corr = df_train[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Heatmap Korelasi Fitur Cuaca", fontweight='bold')
plt.show()


---
## Step 4: Feature Selection & Preprocessing
Kita akan:
1. Menentukan fitur input ($X$) dan target ($y$).
   - Fitur meteorologi: `temperature_2m`, `relative_humidity_2m`, `apparent_temperature`, `surface_pressure`, `cloud_cover`, `wind_speed_10m`, `wind_direction_10m`, `precipitation`, `rain`.
   - Menambahkan ekstraksi waktu: `hour` dan `month` dari kolom `timestamp`.
   - *Catatan*: Kolom `rain_sum_next_6h` **tidak boleh** digunakan sebagai fitur karena merupakan data leakage dari masa depan.
2. Standardisasi fitur numerik menggunakan `StandardScaler` agar model Linear (Logistic Regression) dan Neural Network (TensorFlow) dapat konvergen dengan optimal.


In [ ]:
def extract_features(df):
    data = df.copy()
    # Ekstraksi komponen jam dan bulan dari timestamp
    dt = pd.to_datetime(data['timestamp'])
    data['hour'] = dt.dt.hour
    data['month'] = dt.dt.month
    
    feature_cols = [
        'temperature_2m', 'relative_humidity_2m', 'apparent_temperature',
        'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',
        'precipitation', 'rain', 'hour', 'month'
    ]
    return data[feature_cols], data['rain_next_6h']

X_train_raw, y_train = extract_features(df_train)
X_val_raw, y_val = extract_features(df_valid)
X_test_raw, y_test = extract_features(df_test)

print(f"Fitur yang digunakan ({len(X_train_raw.columns)} fitur): {list(X_train_raw.columns)}")


In [ ]:
# Opsional: Jika ingin running cepat untuk eksperimen awal, atur SAMPLE_SIZE (misal: 50000 atau None untuk full data)
SAMPLE_SIZE = None  # None = gunakan seluruh 147k data train

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(X_train_raw):
    sample_idx = np.random.choice(len(X_train_raw), size=SAMPLE_SIZE, replace=False)
    X_train_exp = X_train_raw.iloc[sample_idx]
    y_train_exp = y_train.iloc[sample_idx]
    print(f"Menggunakan sample training: {SAMPLE_SIZE:,} baris")
else:
    X_train_exp = X_train_raw
    y_train_exp = y_train
    print(f"Menggunakan full dataset training: {len(X_train_exp):,} baris")

# Standardisasi fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_exp)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

print("StandardScaler selesai diterapkan.")


---
## Step 5: Model 1 - Logistic Regression (ML Biasa 1)
Logistic Regression adalah model linear klasik yang sering digunakan sebagai baseline klasifikasi biner. Model ini cepat dilatih dan koefisiennya mudah diinterpretasi.


In [ ]:
print("--- Melatih Model 1: Logistic Regression ---")
start_time = time.time()

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train_exp)

time_lr = time.time() - start_time
print(f"Waktu training Logistic Regression: {time_lr:.2f} detik")

# Prediksi pada Test Set
y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

# Hitung metrik
acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print(f"Accuracy  : {acc_lr:.4f}")
print(f"Precision : {prec_lr:.4f}")
print(f"Recall    : {rec_lr:.4f}")
print(f"F1-Score  : {f1_lr:.4f}")
print(f"ROC-AUC   : {auc_lr:.4f}")


In [ ]:
# Visualisasi Koefisien Logistic Regression
coef_df = pd.DataFrame({
    'Feature': X_train_raw.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(9, 5))
colors = ['#E94E77' if c > 0 else '#4A90E2' for c in coef_df['Coefficient']]
sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette=colors)
plt.title("Koefisien Fitur - Logistic Regression (Merah = Memperbesar Peluang Hujan)", fontweight='bold')
plt.axvline(0, color='black', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


---
## Step 6: Model 2 - Random Forest Classifier (ML Biasa 2)
Random Forest adalah model ensemble berbasis pohon keputusan (*decision trees*) yang mampu menangkap interaksi non-linear yang kompleks antar fitur tanpa memerlukan asumsi linearitas.


In [ ]:
print("--- Melatih Model 2: Random Forest Classifier ---")
start_time = time.time()

# Menggunakan n_jobs=-1 agar memanfaatkan seluruh core CPU
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    min_samples_split=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_exp, y_train_exp)

time_rf = time.time() - start_time
print(f"Waktu training Random Forest: {time_rf:.2f} detik")

# Prediksi pada Test Set
y_pred_rf = rf_model.predict(X_test_raw)
y_prob_rf = rf_model.predict_proba(X_test_raw)[:, 1]

# Hitung metrik
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"Accuracy  : {acc_rf:.4f}")
print(f"Precision : {prec_rf:.4f}")
print(f"Recall    : {rec_rf:.4f}")
print(f"F1-Score  : {f1_rf:.4f}")
print(f"ROC-AUC   : {auc_rf:.4f}")


In [ ]:
# Feature Importance dari Random Forest
feat_imp = pd.DataFrame({
    'Feature': X_train_raw.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis')
plt.title("Tingkat Kepentingan Fitur (Feature Importance) - Random Forest", fontweight='bold')
plt.tight_layout()
plt.show()


---
## Step 7: Model 3 - Deep Neural Network (TensorFlow / Keras)
Model ketiga dibangun menggunakan arsitektur **Deep Neural Network (MLP)** dengan **TensorFlow & Keras**:
- **Layer Arsitektur**:
  - `Dense(128, activation='relu')` + `BatchNormalization()` + `Dropout(0.3)`
  - `Dense(64, activation='relu')` + `Dropout(0.2)`
  - `Dense(32, activation='relu')`
  - `Dense(1, activation='sigmoid')` (Output binary probability)
- **Optimizer**: Adam dengan learning rate `0.001`
- **Loss Function**: Binary Crossentropy
- **Callbacks**:
  - `EarlyStopping`: Menghentikan training otomatis jika `val_loss` tidak membaik dalam 5 epoch dan mengembalikan bobot terbaik (`restore_best_weights=True`).
  - `ReduceLROnPlateau`: Menurunkan learning rate jika performa stagnan.


In [ ]:
def build_tf_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        
        layers.Dense(32, activation='relu'),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

tf_model = build_tf_model(input_dim=X_train_scaled.shape[1])
tf_model.summary()


In [ ]:
print("--- Melatih Model 3: TensorFlow Deep Neural Network ---")
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

start_time = time.time()
history = tf_model.fit(
    X_train_scaled, y_train_exp,
    validation_data=(X_val_scaled, y_val),
    epochs=25,
    batch_size=256,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)
time_tf = time.time() - start_time
print(f"Waktu training TensorFlow MLP: {time_tf:.2f} detik")


In [ ]:
# Plot Training & Validation Loss and Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history['loss'], label='Train Loss', color='#4A90E2', lw=2)
ax1.plot(history.history['val_loss'], label='Val Loss', color='#E94E77', lw=2)
ax1.set_title('Loss vs Epoch (TensorFlow)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Binary Crossentropy')
ax1.legend()

# Accuracy
ax2.plot(history.history['accuracy'], label='Train Accuracy', color='#4A90E2', lw=2)
ax2.plot(history.history['val_accuracy'], label='Val Accuracy', color='#E94E77', lw=2)
ax2.set_title('Accuracy vs Epoch (TensorFlow)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Prediksi pada Test Set menggunakan TensorFlow Model
y_prob_tf = tf_model.predict(X_test_scaled).flatten()
y_pred_tf = (y_prob_tf >= 0.5).astype(int)

# Hitung metrik
acc_tf = accuracy_score(y_test, y_pred_tf)
prec_tf = precision_score(y_test, y_pred_tf)
rec_tf = recall_score(y_test, y_pred_tf)
f1_tf = f1_score(y_test, y_pred_tf)
auc_tf = roc_auc_score(y_test, y_prob_tf)

print(f"Accuracy  : {acc_tf:.4f}")
print(f"Precision : {prec_tf:.4f}")
print(f"Recall    : {rec_tf:.4f}")
print(f"F1-Score  : {f1_tf:.4f}")
print(f"ROC-AUC   : {auc_tf:.4f}")


---
## Step 8: Head-to-Head Comparison (Bandingkan 3 Model)
Sekarang saatnya membandingkan performa ketiga model secara menyeluruh:
1. **Tabel Ringkasan Metrik**
2. **Grafik Batang Perbandingan Metrik**
3. **Kurva ROC (Receiver Operating Characteristic)**
4. **Side-by-Side Confusion Matrix**


In [ ]:
# Tabel Komparasi Metrik
results_df = pd.DataFrame({
    'Model': [
        'Logistic Regression (ML Biasa 1)',
        'Random Forest (ML Biasa 2)',
        'TensorFlow DNN (Deep Learning)'
    ],
    'Accuracy': [acc_lr, acc_rf, acc_tf],
    'Precision': [prec_lr, prec_rf, prec_tf],
    'Recall': [rec_lr, rec_rf, rec_tf],
    'F1-Score': [f1_lr, f1_rf, f1_tf],
    'ROC-AUC': [auc_lr, auc_rf, auc_tf],
    'Training Time (s)': [time_lr, time_rf, time_tf]
})

print("=== TABEL HASIL KOMPARASI MODEL ===")
display(results_df.style.highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'], color='#D4EDDA'))


In [ ]:
# Visualisasi Bar Chart Perbandingan Metrik
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
df_plot = results_df.melt(id_vars='Model', value_vars=metrics_to_plot, var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
sns.barplot(x='Metric', y='Score', hue='Model', data=df_plot, palette=['#5DADE2', '#58D68D', '#AF7AC5'])
plt.title("Perbandingan Metrik Evaluasi: 3 Model", fontsize=14, fontweight='bold')
plt.ylim(0.5, 1.0)
plt.ylabel("Score")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Visualisasi Gabungan Kurva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_tf, tpr_tf, _ = roc_curve(y_test, y_prob_tf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})', color='#5DADE2', lw=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})', color='#58D68D', lw=2)
plt.plot(fpr_tf, tpr_tf, label=f'TensorFlow DNN (AUC = {auc_tf:.3f})', color='#AF7AC5', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6, label='Random Guess')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Perbandingan Kurva ROC (Receiver Operating Characteristic)', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Visualisasi Side-by-Side Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

models_cm = [
    ('Logistic Regression', y_pred_lr, axes[0]),
    ('Random Forest', y_pred_rf, axes[1]),
    ('TensorFlow DNN', y_pred_tf, axes[2])
]

for title, pred, ax in models_cm:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Tidak Hujan', 'Hujan'],
                yticklabels=['Tidak Hujan', 'Hujan'])
    ax.set_title(f'Confusion Matrix: {title}', fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()


---
## Step 9: Kesimpulan & Analisis Akhir
1. **Trade-off Kecepatan vs Akurasi**:
   - **Logistic Regression**: Paling cepat untuk training dan inferensi, sangat ringan, namun kemampuannya terbatas pada hubungan linear antar variabel.
   - **Random Forest**: Sangat kuat pada data tabular cuaca karena secara alami menangkap *threshold* dan batas non-linear cuaca (seperti kelembaban ekstrem atau titik embun).
   - **TensorFlow Deep Neural Network**: Fleksibel dan mampu mencapai performa tinggi dengan regularisasi (BatchNormalization, Dropout), serta dapat diekstensi dengan arsitektur temporal (LSTM/GRU) bila pola sekuensial waktu dipertimbangkan.

2. **Rekomendasi**:
   - Untuk skenario produksi tabular yang membutuhkan akurasi tinggi dan *out-of-the-box performance*, **Random Forest** sering kali memberikan hasil terbaik dengan sedikit hyperparameter tuning.
   - Untuk pipeline deep learning berskala besar atau bila ingin menggabungkan citra satelit/radar dengan data tabular, **TensorFlow** adalah pilihan fondasi terbaik.
